# Feature Engineering — AAPL

Walks through the same feature/target pipeline that `src/features.py` runs as a script, one stage at a time, so each stage's output can be inspected. The feature logic itself lives in `src/feature_utils.build_features`, shared with `src/predict.py` so training and prediction never drift apart.

As noted in `exploration.ipynb`: chronological order is preserved throughout (nothing is shuffled), and any return or target that would span a session gap (overnight/weekend) is masked to NaN rather than silently mixing non-adjacent bars.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the project's src/ package importable regardless of Jupyter's cwd.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_utils import build_features

import warnings
warnings.filterwarnings("ignore")

## 1. Load Data

In [ ]:
# Load the shared 5-minute intraday snapshot -- the same source file used by
# src/ingest.py and src/features.py.
raw_snapshot_path = project_root / "data" / "raw" / "stock-trend_1mo.parquet"

df = pd.read_parquet(raw_snapshot_path)
df_model = (
    df.xs("AAPL", axis=1, level="Ticker")
    .copy()
    .sort_index()
)

print(f"AAPL data shape: {df_model.shape}")
print(f"Period: {df_model.index.min()} to {df_model.index.max()}")
df_model.head()

## 2. Build Features

Runs the full `build_features` pipeline in one call: candle shape, gap-masked returns, rolling volatility/volume, and technical indicators (RSI, MACD, Bollinger Bands, ATR).

In [ ]:
df_model = build_features(df_model)
df_model[["range_pct", "body_pct"]].describe()

In [ ]:
df_model[["return_5m_pct", "return_15m_pct", "return_30m_pct"]].describe()

In [ ]:
df_model[["volatility_30m", "volume_relative"]].describe()

In [ ]:
df_model[["rsi_14", "macd_diff", "bb_width_pct", "atr_pct"]].describe()

## 3. Target: 30-Minute-Ahead Direction

Mirrors `src/features.py`: the label for a bar is only kept when the timestamp 30 minutes ahead is an actual observed bar from the *same trading day* -- not a slot that only exists because we skipped over an overnight/weekend gap.

In [ ]:
timestamps = df_model.index.to_series()

future_close = df_model["Close"].shift(-6)
future_time = timestamps.shift(-6)

valid_target = (
    (future_time - timestamps).eq(pd.Timedelta(minutes=30))
    & future_time.dt.normalize().eq(timestamps.dt.normalize())
)

df_model["target_up_30m"] = (
    (future_close > df_model["Close"])
    .astype("Int64")
    .where(valid_target)
)
df_model["target_time"] = future_time.where(valid_target)

df_model[["Close", "target_up_30m", "target_time"]].tail(10)

## 4. Assemble the Final Training Table

In [ ]:
feature_cols = [
    "range_pct", "body_pct",
    "return_5m_pct", "return_15m_pct", "return_30m_pct",
    "volatility_30m", "volume_relative",
    "rsi_14", "macd_diff", "bb_width_pct", "atr_pct",
]
target_col = "target_up_30m"

# Drop rows with missing values. NaNs come from two sources only:
#   1. Indicator/rolling warm-up at the start of the series (not enough history yet).
#   2. The last 6 rows, which have no future bar to compute a target from.
training_data = df_model[
    feature_cols + [target_col, "target_time"]
].dropna(subset=feature_cols + [target_col, "target_time"]).copy()

training_data[target_col] = training_data[target_col].astype(int)

print("Rows before filter:", len(df_model))
print("Rows for training:", len(training_data))
training_data.head()

Persisted here for inspection only -- `src/features.py` is the script that actually produces the file `src/train.py` reads.

In [ ]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "aapl_training_1mo.parquet"

training_data.to_parquet(output_path, index=True)
print(f"Saved {training_data.shape[0]} rows x {training_data.shape[1]} columns to {output_path}")

## 5. Sanity Checks

In [ ]:
# Class balance of the target -- important to know before training, since a
# heavily imbalanced target (e.g. 90% "up") would make plain accuracy misleading.
training_data["target_up_30m"].value_counts(normalize=True).rename("share")